In [4]:
from platform import python_version
print(python_version())

3.12.12


### find human skin melanoma single-cell studies

In [ ]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

from scipy.stats import spearmanr

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/rna_velocity")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, str(ROOT_SRC))

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir

root_figure = create_dir(ROOT0, "pictures")
root_results = create_dir(ROOT0, "results")
root_tables = create_dir(root_results, "tables")

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

In [2]:
out = {}

terms = {
    "geo_scrna": '(melanoma[Title] OR melanoma[Description]) AND \
    "Homo sapiens"[Organism] AND gse[ETYP] AND '
    '("single cell"[All Fields] OR "single-cell"[All Fields] OR "single nucleus"[All Fields] OR scRNA[All Fields])',
}
for k, t in terms.items():
    try:
        r = host.mcp("omics-archives", "geo_search_series", term=t, retmax=500)
        out[k] = r
        print(k, r.get("count"), len(r.get("records", [])))
    except Exception as e:
        print(k, "ERR", e)

try:
    ae = host.mcp("omics-archives", "arrayexpress_search_experiments",
                  query="melanoma", organism="Homo sapiens",
                  study_type="RNA-seq of coding RNA from single cells", max_records=200)
    out["ae"] = ae
    print("ae", ae.get("total_hits"), len(ae.get("records", [])))
except Exception as e:
    print("ae ERR", e)

json.dump(out, open("handoff/search_raw.json","w"))

geo_scrna ERR name 'host' is not defined
ae ERR name 'host' is not defined


In [ ]:
import json, pandas as pd
recs = json.load(open("handoff/geo_records.json"))

SYS = """You curate genomics metadata. Given a GEO series record, judge it strictly and answer with ONE JSON object only, no prose.
Fields:
"singlecell": one of "single_cell","single_nucleus","spatial","bulk_only" -- is the assay itself at single-cell/nucleus resolution (10x, Smart-seq, CITE-seq, scATAC, snRNA, Drop-seq, MARS-seq, plate-based), or spatial transcriptomics, or is the series bulk data that merely mentions single-cell?
"melanoma_material": one of "patient_cutaneous","patient_melanoma_site_unclear","patient_uveal","patient_mucosal","cellline_or_pdx","mouse_model","not_melanoma" -- what melanoma material was profiled. Use "patient_cutaneous" for human skin/cutaneous melanoma primaries, cutaneous-origin metastases (lymph node, brain, visceral mets of skin melanoma), or acral melanoma. "patient_melanoma_site_unclear" if human patient melanoma but the primary site (cutaneous vs uveal vs mucosal) cannot be told from the text.
"human": true/false -- human material profiled (true even if mixed human+mouse).
"modalities": short list from ["scRNA","snRNA","scATAC","CITE-seq","TCR/BCR-seq","spatial","perturb-seq","other"].
"context": <=12 words, e.g. "ICI-treated metastatic melanoma, pre/post biopsies".
Answer JSON only."""

reqs = []
for r in recs:
    txt = (f"Accession: {r.get('accession')}\nTitle: {r.get('title')}\nType: {r.get('gdstype')}\n"
           f"Organism: {r.get('taxon')}\nSamples: {r.get('n_samples')}\nDate: {r.get('pdat')}\n"
           f"Summary: {(r.get('summary') or '')[:2200]}")
    reqs.append({"prompt": txt, "system": SYS, "max_tokens": 400})

res = host.llm(reqs, model=host.reasoning_model(), max_concurrency=12)

def parse(x):
    try:
        t = x["text"]
        s = t[t.index("{"): t.rindex("}")+1]
        return json.loads(s)
    except Exception:
        return {}

rows = []
for r, x in zip(recs, res):
    p = parse(x)
    rows.append({"accession": r.get("accession"), "title": r.get("title"),
                 "n_samples": r.get("n_samples"), "date": r.get("pdat"),
                 "gdstype": r.get("gdstype"), "pubmed": ",".join(map(str, r.get("pubmed", []) or [])) if isinstance(r.get("pubmed"), list) else r.get("pubmed"),
                 "singlecell": p.get("singlecell"), "material": p.get("melanoma_material"),
                 "human": p.get("human"), "modalities": ";".join(p.get("modalities", []) or []),
                 "context": p.get("context"), "summary": r.get("summary")})
df = pd.DataFrame(rows)
print(df["singlecell"].value_counts(dropna=False).to_dict())
print(df["material"].value_counts(dropna=False).to_dict())
print(list(recs[0].keys()))

In [ ]:
M = host.reasoning_model()
for q in reqs: q["model"] = M
res = host.llm(reqs, max_concurrency=12)

def parse(x):
    try:
        t = x["text"]; s = t[t.index("{"): t.rindex("}")+1]
        return json.loads(s)
    except Exception:
        return {}

rows = []
for r, x in zip(recs, res):
    p = parse(x)
    pm = r.get("pubmed") or r.get("pubmedids") or []
    rows.append({"accession": r.get("accession"), "title": r.get("title"),
                 "n_samples": r.get("n_samples"), "date": r.get("pdat"),
                 "gdstype": r.get("gdstype"),
                 "pubmed": ";".join(map(str, pm)) if isinstance(pm, list) else str(pm),
                 "singlecell": p.get("singlecell"), "material": p.get("melanoma_material"),
                 "human": p.get("human"), "modalities": ";".join(p.get("modalities") or []),
                 "context": p.get("context"), "summary": r.get("summary")})
df = pd.DataFrame(rows)
print(df["singlecell"].value_counts(dropna=False).to_dict())
print(df["material"].value_counts(dropna=False).to_dict())

In [ ]:
fail_idx = [i for i,r in enumerate(rows) if r["singlecell"] is None]
res2 = host.llm([reqs[i] for i in fail_idx], max_concurrency=6)
for i, x in zip(fail_idx, res2):
    p = parse(x)
    df.loc[i, ["singlecell","material","human","modalities","context"]] = [
        p.get("singlecell"), p.get("melanoma_material"), p.get("human"),
        ";".join(p.get("modalities") or []), p.get("context")]
print(df["singlecell"].isna().sum(), df["material"].value_counts(dropna=False).to_dict())
landmarks = ["GSE72056","GSE115978","GSE120575","GSE123139","GSE189889","GSE215120","GSE159251"]
print({g: g in set(df.accession) for g in landmarks})

In [ ]:
sc_ok = {"single_cell","single_nucleus","spatial","scATAC"}
d = df.copy()
d["singlecell"] = d["singlecell"].fillna("unknown")
d["material"] = d["material"].fillna("unknown")
d["year"] = d["date"].str.slice(0,4)

pat = d[(d.human == True) & d.singlecell.isin(sc_ok) &
        d.material.isin(["patient_cutaneous","patient_melanoma_site_unclear"])].copy()
pat["tier"] = pat.material.map({"patient_cutaneous":"A_cutaneous_confirmed",
                                "patient_melanoma_site_unclear":"B_site_not_stated"})
pat["geo_url"] = "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=" + pat.accession
pat = pat.sort_values(["tier","year","accession"], ascending=[True,False,True])
cols = ["accession","tier","year","date","title","context","singlecell","modalities",
        "n_samples","gdstype","pubmed","geo_url","summary"]
pat[cols].to_csv("melanoma_singlecell_geo_studies.csv", index=False)

# excluded-but-adjacent pools, for the record
other = d[d.singlecell.isin(sc_ok) & ~d.index.isin(pat.index) &
          d.material.isin(["cellline_or_pdx","patient_uveal","patient_mucosal","mouse_model"])].copy()
other["geo_url"] = "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=" + other.accession
other.sort_values(["material","year"], ascending=[True,False])[
    ["accession","material","year","title","context","singlecell","modalities","n_samples","pubmed","geo_url"]
].to_csv("melanoma_singlecell_adjacent_pools.csv", index=False)

print(len(pat), pat.tier.value_counts().to_dict())
print(pat.year.value_counts().sort_index().to_dict())
print(pat.modalities.str.get_dummies(";").sum().sort_values(ascending=False).to_dict())
print(len(other), other.material.value_counts().to_dict())
print(pat.nlargest(15,"n_samples")[["accession","year","n_samples","context"]].to_string(index=False))



#### Search on 2026/09/24

![melanoma production](../claude_results/melanoma_sc_studies_by_year.png)

### filter tumor tissue, skin melanoma, with genome data available, able to run scVelo

In [ ]:
cand = sub[(sub.tissue_based == True) & sub.singlecell.isin(["single_cell","single_nucleus","scATAC","spatial"])].copy()
cand = cand.reset_index(drop=True)
print(len(cand), cand.tier.value_counts().to_dict())

SYS3 = """Given a GEO series record, reply with ONE JSON object only:
"platform": one of "10x_3prime","10x_5prime","10x_multiome","other_droplet","plate_smartseq","spatial_only","unclear" -- the single-cell chemistry used for the transcriptome (pick the main one; 10x_5prime for 5' GEX usually paired with TCR).
"umi_based": true if a droplet/UMI protocol (10x, Drop-seq, inDrop, BD Rhapsody, Seq-Well), false for plate/full-length (Smart-seq2, CEL-seq), null if unclear.
"cutaneous_certainty": "explicit" (text says cutaneous/skin/acral primary), "implied" (metastatic melanoma with no mention of uveal/mucosal, so cutaneous by default), "unclear".
"mentions_raw_restricted": true if the text says raw data are in dbGaP/EGA/controlled access or available on request.
"mentions_velocity": true if RNA velocity / velocyto / spliced-unspliced is mentioned.
JSON only."""
rq3 = [{"prompt": f"Accession: {r.accession}\nTitle: {r.title}\nType: {r.gdstype}\nSamples: {r.n_samples}\nSummary: {(r.summary or '')[:2400]}",
        "system": SYS3, "model": M, "max_tokens": 250} for r in cand.itertuples()]
r4 = host.llm(rq3, max_concurrency=12)
p4 = [parse(x) for x in r4]
for k in ["platform","umi_based","cutaneous_certainty","mentions_raw_restricted","mentions_velocity"]:
    cand[k] = [p.get(k) for p in p4]
print(cand.platform.value_counts(dropna=False).to_dict())
print(cand.cutaneous_certainty.value_counts(dropna=False).to_dict())
print(cand.umi_based.value_counts(dropna=False).to_dict())



In [7]:


EMAIL = 'flalix@gmail.com'

base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
def eutil(path, **kw):
    if EMAIL: kw["email"] = EMAIL
    kw["tool"] = "science-agent"
    return requests.get(base + path, params=kw, timeout=60)

sra_info = {}
for acc in cand.accession:
    rec = {"sra_uids": 0, "srp": [], "err": None}
    try:
        s = eutil("esearch.fcgi", db="sra", term=f"{acc}[All Fields]", retmax=200, retmode="json").json()
        ids = s["esearchresult"].get("idlist", [])
        rec["sra_uids"] = int(s["esearchresult"].get("count", 0))
        if ids:
            su = eutil("esummary.fcgi", db="sra", id=",".join(ids[:20]), retmode="json").json()
            srps = set()
            for k, v in su.get("result", {}).items():
                if k == "uids": continue
                m = re.search(r'Study acc="((?:SRP|ERP|DRP)\d+)"', v.get("expxml", "") or "")
                if m: srps.add(m.group(1))
            rec["srp"] = sorted(srps)
    except Exception as e:
        rec["err"] = repr(e)[:120]
    sra_info[acc] = rec
    time.sleep(0.34)
json.dump(sra_info, open("handoff/sra_info.json","w"))
print(sum(1 for v in sra_info.values() if v["srp"]), "of", len(sra_info), "have a public SRA study")
print([a for a,v in sra_info.items() if not v["srp"]])



NameError: name 'cand' is not defined

#### Results

1 of 52 have a public SRA study
['GSE242941', 'GSE294274', 'GSE335480', 'GSE344166', 'GSE277165', 'GSE286410', 'GSE244983', 'GSE254918', 'GSE229908', 'GSE242477', 'GSE185386', 'GSE189889', 'GSE200215', 'GSE200218', 'GSE215120', 'GSE215121', 'GSE221390', 'GSE174401', 'GSE115978', 'GSE118478', 'GSE72056', 'GSE294273', 'GSE320040', 'GSE269936', 'GSE283942', 'GSE291246', 'GSE295942', 'GSE303948', 'GSE221553', 'GSE222446', 'GSE222750', 'GSE261023', 'GSE262779', 'GSE275929', 'GSE216756', 'GSE229858', 'GSE229860', 'GSE234352', 'GSE198265', 'GSE216055', 'GSE138720', 'GSE159251', 'GSE179373', 'GSE186344', 'GSE134388', 'GSE140819', 'GSE158803', 'GSE127471', 'GSE139249', 'GSE111360', 'GSE123139']

In [ ]:
print({a: v["sra_uids"] for a,v in list(sra_info.items())[:6]}, [v["err"] for v in list(sra_info.values())[:3]])
t = eutil("esearch.fcgi", db="sra", term="GSE115978", retmax=5, retmode="json").json()
print("plain term:", t["esearchresult"].get("count"), t["esearchresult"].get("idlist"), t["esearchresult"].get("errorlist"))
# ENA route
u = "https://www.ebi.ac.uk/ena/portal/api/search"
p = {"result":"read_run","query":'study_alias="GSE115978"',
     "fields":"run_accession,study_accession,secondary_study_accession,library_strategy,instrument_model,fastq_ftp,submitted_format,library_layout",
     "format":"tsv","limit":3}
r = requests.get(u, params=p, timeout=60)
print("ENA", r.status_code, r.text[:600])

#### Results

{'GSE242941': 0, 'GSE294274': 0, 'GSE335480': 0, 'GSE344166': 0, 'GSE277165': 0, 'GSE286410': 0} [None, None, None]
plain term: 0 [] {'phrasesnotfound': ['GSE115978'], 'fieldsnotfound': []}
ENA 200 run_accession	study_accession	secondary_study_accession	library_strategy	instrument_model	fastq_ftp	submitted_format	library_layout

#### Fetching GEO SOFT headers for raw-data relations

In [ ]:
ctrl = eutil("esearch.fcgi", db="sra", term="GSE164378", retmax=3, retmode="json").json()
print("control GSE164378 in SRA:", ctrl["esearchresult"].get("count"))

def soft(acc):
    r = requests.get("https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi",
                     params={"acc": acc, "targ": "self", "form": "text", "view": "brief"}, timeout=90)
    return r.text

soft_txt = {}
for acc in cand.accession:
    try:
        soft_txt[acc] = soft(acc)
    except Exception as e:
        soft_txt[acc] = "ERR " + repr(e)[:100]
    time.sleep(0.34)
json.dump(soft_txt, open("handoff/soft_headers.json","w"))
rel = {a: [l.split("=",1)[1].strip() for l in t.splitlines() if l.startswith("!Series_relation")] for a,t in soft_txt.items()}
from collections import Counter
print(Counter(w for v in rel.values() for w in set(x.split(":")[0] for x in v)))
print(sum(1 for v in rel.values() if any(x.startswith("SRA") for x in v)), "with SRA relation")
print({a:v for a,v in list(rel.items())[:4]})

### Checking ENA and SRA run avaiability per BioProject

In [ ]:
import io


In [ ]:


prj = {}
for a, t in soft_txt.items():
    m = re.findall(r"bioproject/(PRJ[A-Z]+\d+)", t)
    prj[a] = sorted(set(m))
dbgap = {a: sorted(set(re.findall(r"(phs\d{6})", t))) for a, t in soft_txt.items()}

ENA_FIELDS = ("run_accession,study_accession,secondary_study_accession,experiment_title,"
              "library_strategy,library_layout,instrument_model,fastq_ftp,submitted_ftp,read_count,base_count")
def ena_runs(p):
    r = requests.get("https://www.ebi.ac.uk/ena/portal/api/search",
        params={"result":"read_run","query":f'study_accession="{p}"',"fields":ENA_FIELDS,
                "format":"tsv","limit":0}, timeout=120)
    if r.status_code != 200 or not r.text.strip(): return pd.DataFrame()
    return pd.read_csv(io.StringIO(r.text), sep="\t")

ena = {}
for a in cand.accession:
    frames = []
    for p in prj[a]:
        try:
            d = ena_runs(p)
            if len(d): frames.append(d)
        except Exception as e:
            pass
        time.sleep(0.2)
    ena[a] = pd.concat(frames) if frames else pd.DataFrame()

summ = []
for a in cand.accession:
    d = ena[a]
    nf = int(d.fastq_ftp.notna().sum()) if len(d) and "fastq_ftp" in d else 0
    ns = int(d.submitted_ftp.notna().sum()) if len(d) and "submitted_ftp" in d else 0
    tb = float(d.base_count.sum())/1e12 if len(d) and "base_count" in d else 0.0
    strat = ";".join(sorted(set(d.library_strategy.dropna()))) if len(d) else ""
    inst = ";".join(sorted(set(d.instrument_model.dropna())))[:60] if len(d) else ""
    titles = " | ".join(list(dict.fromkeys(d.experiment_title.dropna()))[:3])[:180] if len(d) else ""
    summ.append({"accession": a, "bioproject": ";".join(prj[a]), "dbgap": ";".join(dbgap[a]),
                 "ena_runs": len(d), "runs_with_fastq": nf, "runs_with_submitted": ns,
                 "total_Tbases": round(tb,3), "ena_strategy": strat, "ena_instrument": inst,
                 "ena_titles": titles})
av = pd.DataFrame(summ)
print(av.runs_with_fastq.gt(0).sum(), "of 52 with public FASTQ in ENA")
print(av[av.runs_with_fastq>0][["accession","ena_runs","runs_with_fastq","total_Tbases","ena_strategy"]].to_string(index=False))
print("dbGaP-flagged:", av[av.dbgap!=""].accession.tolist())

27 of 52 with public FASTQ in ENA  
accession  ena_runs  runs_with_fastq  total_Tbases          ena_strategy  
GSE277165        74               74         2.408      ATAC-seq;RNA-Seq  
GSE286410        41               41         0.307         OTHER;RNA-Seq  
GSE242477        17               17         0.459         OTHER;RNA-Seq  
GSE189889        68               68         0.127               RNA-Seq  
GSE215121        57               57         0.384               RNA-Seq  
GSE221390        28               28         0.120               RNA-Seq  
GSE174401       280              280         0.198               RNA-Seq  
GSE311199        78               20         5.976 ATAC-seq;Hi-C;RNA-Seq  
GSE269936       179              179         1.677               RNA-Seq  
GSE303948        14               14         0.001              ATAC-seq  
GSE221553        22               22         0.291               RNA-Seq  
GSE222446        13               13         0.224               RNA-Seq  
GSE222750        22               22         0.074               RNA-Seq  
GSE261023         3                3         0.059                 OTHER  
GSE262779        10               10         0.061               RNA-Seq  
GSE275929       141              141         3.282         OTHER;RNA-Seq  
GSE229858        13               13         0.172               RNA-Seq  
GSE229860        13               13         0.046                 OTHER  
GSE234352        65               65         0.029                 OTHER  
GSE198265        22               22         1.620               RNA-Seq  
GSE159251        60               60         1.213               RNA-Seq  
GSE179373        44               44         0.396               RNA-Seq  
GSE186344         8                8         0.079               RNA-Seq  
GSE134388        21               20         0.065       RIP-Seq;RNA-Seq  
GSE158803        24               24         0.233               RNA-Seq  
GSE139249        18               18         0.198               RNA-Seq  
GSE111360        44               44         3.241         OTHER;RNA-Seq  
dbGaP-flagged: []  
  
Merging availability flags and detecting loom files


#### Mergint avaiability flags and detecting loom files

In [ ]:
c2 = cand.merge(av, on="accession", how="left")
# fastq file-count evidence (R1 barcode read present?)
fq = {}
for a in cand.accession:
    d = ena[a]
    if len(d) and "fastq_ftp" in d:
        n = d.fastq_ftp.dropna().map(lambda s: len(str(s).split(";")))
        fq[a] = (int(n.max()) if len(n) else 0, round(float(n.mean()),2) if len(n) else 0)
    else:
        fq[a] = (0, 0)
c2["fastq_files_max"] = [fq[a][0] for a in c2.accession]
c2["fastq_files_mean"] = [fq[a][1] for a in c2.accession]
c2["supp_loom"] = [bool(re.search(r"loom|velocyto|spliced", soft_txt[a], re.I)) for a in c2.accession]
c2["supp_bam"] = [bool(re.search(r"\.bam", soft_txt[a], re.I)) for a in c2.accession]

droplet = c2.platform.isin(["10x_3prime","10x_5prime","10x_multiome","other_droplet"]) | (c2.umi_based == True)
pub = c2.runs_with_fastq > 0
print("droplet:", int(droplet.sum()), "| public fastq:", int(pub.sum()), "| both:", int((droplet & pub).sum()))
sh = c2[droplet & pub].copy()
print(sh[["accession","tier","year","platform","cutaneous_certainty","runs_with_fastq","fastq_files_max",
          "total_Tbases","supp_loom","specimen","context"]].to_string(index=False))
print("\nunclear-platform but public fastq:", c2[~droplet & pub].accession.tolist())

droplet: 27 | public fastq: 27 | both: 15  
accession                  tier year      platform cutaneous_certainty  runs_with_fastq  fastq_files_max  total_Tbases  supp_loom                                       specimen                                                                           context  
GSE277165 A_cutaneous_confirmed 2025    10x_3prime             implied               74                2         2.408      False                             primary_skin_tumor      Primary human melanoma, high/intermediate/low immune infiltration categories  
GSE286410 A_cutaneous_confirmed 2025 other_droplet            explicit               41                2         0.307      False       primary_skin_tumor;metastasis_lymph_node           Paediatric cutaneous melanoma, primary tumour and lymph node metastases  
GSE242477 A_cutaneous_confirmed 2023    10x_5prime             unclear               17                2         0.459      False            primary_skin_tumor;metastasis_other         Acral melanoma TILs, single-cell phenotyping and TCR reactivity screening  
GSE189889 A_cutaneous_confirmed 2022    10x_3prime            explicit               68                1         0.127      False                             primary_skin_tumor              Acral melanoma tumors, single-cell RNA-seq for immunotherapy targets  
GSE303948     B_site_not_stated 2025 other_droplet             implied               14                2         0.001      False                               metastasis_other                         snATAC-seq of cDCs from human metastatic melanoma samples  
GSE221553     B_site_not_stated 2024    10x_3prime             implied               22                1         0.291      False                               metastasis_other              Metastatic melanoma, TIL-ACT therapy, pre/post biopsies, CD45+ cells  
GSE222446     B_site_not_stated 2024    10x_3prime             implied               13                1         0.224      False                               metastasis_other                  Metastatic melanoma, pre/post TIL adoptive cell therapy biopsies  
GSE222750     B_site_not_stated 2024    10x_5prime             implied               22                1         0.074      False                               metastasis_other       Metastatic melanoma, TIL adoptive cell therapy, pre/post treatment biopsies  
GSE261023     B_site_not_stated 2024  10x_multiome             implied                3                1         0.059      False                    metastasis_other;blood_PBMC        TIL therapy in melanoma, multiome tracking clonotypes baseline to post-ACT  
GSE262779     B_site_not_stated 2024    10x_3prime             implied               10                2         0.061      False                               metastasis_other              Melanoma FNAs pre/during BRAFi/MEKi therapy, resistant vs responding  
GSE229858     B_site_not_stated 2023    10x_5prime             implied               13                2         0.172      False primary_skin_tumor;metastasis_other;blood_PBMC                TIL therapy in melanoma, baseline/product/post-ACT blood and tumor  
GSE229860     B_site_not_stated 2023    10x_5prime             implied               13                1         0.046      False                    metastasis_other;blood_PBMC            TIL-ACT melanoma patients, baseline tumor/product/post-ACT blood-tumor  
GSE198265     B_site_not_stated 2022    10x_5prime             implied               22                2         1.620      False                               metastasis_other Human melanoma tumor-infiltrating CD4+/CD8+ T cells, neoantigen specificity study  
GSE159251     B_site_not_stated 2021    10x_5prime             implied               60                2         1.213      False                    metastasis_other;blood_PBMC                 Advanced metastatic melanoma CD8+ T cells, tumor and paired blood  
GSE111360     B_site_not_stated 2018    10x_5prime             unclear               44                2         3.241      False      primary_skin_tumor;metastasis_other;other Patient-derived tumor organoids incl. melanoma resections, anti-PD-1 ICB response  
  
unclear-platform but public fastq: ['GSE215121', 'GSE221390', 'GSE174401', 'GSE311199', 'GSE269936', 'GSE275929', 'GSE234352', 'GSE179373', 'GSE186344', 'GSE134388', 'GSE158803', 'GSE139249']

#### Fetching sample-level GEO metadata for public-raw studies

In [ ]:
targets = av[av.runs_with_fastq > 0].accession.tolist()
gsm_txt = {}
for a in targets:
    try:
        r = requests.get("https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi",
                         params={"acc": a, "targ": "gsm", "form": "text", "view": "brief"}, timeout=180)
        gsm_txt[a] = r.text
    except Exception as e:
        gsm_txt[a] = "ERR " + repr(e)[:100]
    time.sleep(0.4)
json.dump(gsm_txt, open("handoff/gsm_soft.json","w"))

def parse_gsm(t):
    blocks = re.split(r"(?=\^SAMPLE = )", t)
    rows = []
    for b in blocks:
        if not b.startswith("^SAMPLE"): continue
        g = {"gsm": re.search(r"\^SAMPLE = (GSM\d+)", b).group(1)}
        g["title"] = (re.search(r"!Sample_title = (.*)", b) or [None,""])[1] if re.search(r"!Sample_title = (.*)", b) else ""
        g["srx"] = re.findall(r"SRA: https://www\.ncbi\.nlm\.nih\.gov/sra\?term=(SRX\d+)", b)
        g["strategy"] = ";".join(re.findall(r"!Sample_library_strategy = (.*)", b))
        g["instr"] = ";".join(re.findall(r"!Sample_instrument_model = (.*)", b))
        g["supp"] = ";".join(re.findall(r"!Sample_supplementary_file[^=]*= (.*)", b))
        g["type"] = ";".join(re.findall(r"!Sample_type = (.*)", b))
        rows.append(g)
    return pd.DataFrame(rows)

gsm = {a: parse_gsm(t) for a, t in gsm_txt.items()}
prof = []
for a, d in gsm.items():
    supp = " ".join(d.supp).lower() if len(d) else ""
    prof.append({"accession": a, "n_gsm": len(d),
                 "n_gsm_with_srx": int(d.srx.map(bool).sum()) if len(d) else 0,
                 "srx_set": sorted({x for l in d.srx for x in l}) if len(d) else [],
                 "tenx_supp": bool(re.search(r"barcodes\.tsv|features\.tsv|genes\.tsv|matrix\.mtx|filtered_feature_bc|molecule_info", supp)),
                 "loom_supp": bool(re.search(r"\.loom|velocyto|spliced", supp)),
                 "bam_supp": ".bam" in supp,
                 "strategies": ";".join(sorted({s for s in d.strategy if s})) if len(d) else "",
                 "instruments": ";".join(sorted({s for s in d.instr if s}))[:70] if len(d) else ""})
pf = pd.DataFrame(prof)
print(pf[["accession","n_gsm","n_gsm_with_srx","tenx_supp","loom_supp","bam_supp","strategies"]].to_string(index=False))

#### Results

accession  n_gsm  n_gsm_with_srx  tenx_supp  loom_supp  bam_supp                  strategies  
GSE277165     20              20       True      False     False        ATAC-seq\r;RNA-Seq  
GSE286410     41              41      False      False     False           OTHER\r;RNA-Seq  
GSE242477     17              17       True      False     False           OTHER\r;RNA-Seq  
GSE189889      9               9      False      False     False                   RNA-Seq  
GSE215121     68              68       True      False     False                   RNA-Seq  
GSE221390      8               8       True      False     False                   RNA-Seq  
GSE174401     43              43      False      False     False                   RNA-Seq  
GSE311199     15              15      False      False     False ATAC-seq\r;Hi-C\r;RNA-Seq  
GSE269936     38              38       True      False     False                   RNA-Seq  
GSE303948     14              14       True      False     False                  ATAC-seq  
GSE221553     22              22      False      False     False                   RNA-Seq  
GSE222446     12              12      False      False     False                   RNA-Seq  
GSE222750     22              22      False      False     False                   RNA-Seq  
GSE261023      3               3       True      False     False                     OTHER  
GSE262779     10              10      False      False     False                   RNA-Seq  
GSE275929     58              58       True      False     False           OTHER\r;RNA-Seq  
GSE229858     13              13      False      False     False                   RNA-Seq  
GSE229860     13              13      False      False     False                     OTHER  
GSE234352     63              63      False      False     False                     OTHER  
GSE198265     22              22      False      False     False                   RNA-Seq  
GSE159251     15              15      False      False     False                   RNA-Seq  
GSE179373     21              21       True      False     False                   RNA-Seq  
GSE186344     18               2       True      False     False                   RNA-Seq  
GSE134388     21              21       True      False     False         RIP-Seq\r;RNA-Seq  
GSE158803     10              10       True      False     False                   RNA-Seq  
GSE139249     12              12       True      False     False                   RNA-Seq  
GSE111360     35              35       True      False     False           OTHER\r;RNA-Seq  

### Resolving chemistry from protocols and linking runs to samples

In [ ]:
quick = {}
for a in targets:
    try:
        quick[a] = requests.get("https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi",
            params={"acc": a, "targ": "gsm", "form": "text", "view": "quick"}, timeout=240).text
    except Exception as e:
        quick[a] = "ERR " + repr(e)[:100]
    time.sleep(0.4)
json.dump(quick, open("handoff/gsm_quick.json","w"))

pats = {"10x": r"10x genomics|10X Genomics|chromium|cellranger|cell ranger",
        "v3": r"single cell 3'|3' v[23]|Single Cell 3", "v5": r"5' ?(gex|v[12])|single cell 5|VDJ|V\(D\)J",
        "smartseq": r"smart-?seq", "multiome": r"multiome|ATAC \+ Gene Expression",
        "snrna": r"single[- ]nucle", "parse": r"parse biosciences|evercode|BD Rhapsody|Seq-Well|inDrop|Drop-seq"}
chem = []
for a in targets:
    t = quick[a]
    chem.append({"accession": a, **{k: bool(re.search(v, t, re.I)) for k, v in pats.items()}})
ch = pd.DataFrame(chem)

ENA_F2 = "run_accession,experiment_accession,library_strategy,fastq_ftp,submitted_ftp,read_count,base_count,instrument_model"
ena2 = {}
for a in targets:
    frames = []
    for p in prj[a]:
        try:
            r = requests.get("https://www.ebi.ac.uk/ena/portal/api/search",
                params={"result":"read_run","query":f'study_accession="{p}"',"fields":ENA_F2,
                        "format":"tsv","limit":0}, timeout=120)
            if r.status_code == 200 and r.text.strip():
                frames.append(pd.read_csv(io.StringIO(r.text), sep="\t"))
        except Exception: pass
        time.sleep(0.2)
    ena2[a] = pd.concat(frames) if frames else pd.DataFrame()

link = []
for a in targets:
    d, srx = ena2[a], set(pf.set_index("accession").loc[a, "srx_set"])
    if len(d) and srx:
        dd = d[d.experiment_accession.isin(srx)]
    else:
        dd = d.iloc[0:0] if len(d) else pd.DataFrame()
    rna = dd[dd.library_strategy.isin(["RNA-Seq","OTHER"])] if len(dd) else dd
    nfiles = rna.fastq_ftp.dropna().map(lambda s: len(str(s).split(";"))) if len(rna) else pd.Series(dtype=int)
    link.append({"accession": a, "runs_linked": len(dd), "rna_runs_linked": len(rna),
                 "rna_runs_with_fastq": int(rna.fastq_ftp.notna().sum()) if len(rna) else 0,
                 "files_per_run_mode": int(nfiles.mode().iloc[0]) if len(nfiles) else 0,
                 "files_per_run_max": int(nfiles.max()) if len(nfiles) else 0,
                 "rna_Tbases": round(float(rna.base_count.sum())/1e12, 3) if len(rna) else 0.0})
lk = pd.DataFrame(link)
print(lk.merge(ch, on="accession")[["accession","runs_linked","rna_runs_linked","files_per_run_mode",
      "files_per_run_max","rna_Tbases","10x","v3","v5","multiome","smartseq","parse","snrna"]].to_string(index=False))



NameError: name 'targets' is not defined

### can you filter which has immunotherapy with + and - responses

In [11]:
root_result = "/home/flavio/uv/rna_velocity/claude_results"
os.listdir(root_result)

['melanoma_scvelo_ready_studies.csv',
 'melanoma_singlecell_adjacent_pools.csv',
 'melanoma_sc_studies_by_year.png',
 'melanoma_singlecell_geo_studies.csv',
 'melanoma_scvelo_excluded_with_reasons.csv',
 '.~lock.melanoma_scvelo_excluded_with_reasons.csv#']

#### Scanning sample characteristics for response annotation

In [13]:
fname = 'melanoma_scvelo_ready_studies.csv'
dfstu = pd.read_csv(os.path.join(root_result, fname))
print(dfstu.shape)
print(dfstu.columns)
dfstu.head(3)

(16, 20)
Index(['accession', 'tier', 'cutaneous', 'year', 'context', 'specimen',
       'sorting', 'scvelo_target', 'melanoma_fraction', 'platform', 'n_gsm',
       'rna_runs_public', 'raw_Tbases', 'read_structure', 'fastq_route',
       'note', 'bioproject', 'pubmed', 'geo_url', 'title'],
      dtype='object')


,accession,tier,cutaneous,year,context,specimen,sorting,scvelo_target,melanoma_fraction,platform,n_gsm,rna_runs_public,raw_Tbases,read_structure,fastq_route,note,bioproject,pubmed,geo_url,title
0,GSE269936,B_site_not_stated,implied,2025,"Metastatic melanoma, scRNA-seq, heterogeneity ...",metastasis_lymph_node;metastasis_other,whole_tumor_unsorted,malignant + immune,all,unclear,38,179,1.677,"R0:8,R1:26,R2:57",SRA only: fasterq-dump --include-technical --s...,Whole dissociated metastatic melanoma tumor ce...,PRJNA1124138,40890106,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Gene expression profile at single cell level f...
1,GSE222446,B_site_not_stated,implied,2024,"Metastatic melanoma, pre/post TIL adoptive cel...",metastasis_other,whole_tumor_unsorted,malignant + immune,all,10x_3prime,12,13,0.224,"R0:28,R1:91,R2:10,R3:10",SRA only: fasterq-dump --include-technical --s...,Dissociated whole-tumor scRNA-seq from metasta...,PRJNA922121,38306416;38714897,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Response to tumor- infiltrating lymphocyte ado...
2,GSE174401,A_cutaneous_confirmed,implied,2021,"Melanoma brain/leptomeningeal metastases, CSF ...",metastasis_brain;metastasis_other,whole_tumor_unsorted,malignant + immune,all,unclear,43,280,0.198,"R0:8,R1:26,R2:98",SRA only: fasterq-dump --include-technical --s...,CSF and brain/skin metastasis cells; immune mi...,PRJNA729750,34035069;38030596,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Single cell characterization of the immune mic...


In [14]:
keep16 = dfstu.accession.tolist()
def chars(a):
    blocks = re.split(r"(?=\^SAMPLE = )", quick[a])
    rows = []
    for b in blocks:
        if not b.startswith("^SAMPLE"): continue
        g = re.search(r"\^SAMPLE = (GSM\d+)", b).group(1)
        cc = re.findall(r"!Sample_characteristics_ch1 = (.*)", b)
        ti = re.findall(r"!Sample_title = (.*)", b)
        rows.append({"gsm": g, "title": (ti[0].strip() if ti else ""), "chars": [x.strip() for x in cc]})
    return rows

RESP = re.compile(r"respon|recist|\bPD\b|\bSD\b|\bPR\b|\bCR\b|progress|benefit|outcome|refractor|resist|relaps|survival|efficac", re.I)
TREAT = re.compile(r"anti-?pd-?1|pembrolizum|nivolumab|ipilimumab|anti-?ctla|checkpoint|\bICI\b|\bICB\b|immunotherap|TIL|adoptive|BRAF|MEK|dabrafenib|trametinib|vemurafenib|interleukin|IL-2", re.I)
summary_rows = []
for a in keep16:
    rows = chars(a)
    keys = {}
    for r in rows:
        for c in r["chars"]:
            k = c.split(":")[0].strip().lower()
            keys.setdefault(k, set()).add(c.split(":",1)[1].strip() if ":" in c else "")
    resp_keys = {k: sorted(v)[:8] for k, v in keys.items() if RESP.search(k)}
    treat_keys = {k: sorted(v)[:8] for k, v in keys.items() if TREAT.search(k) or "treat" in k or "therap" in k}
    title_resp = sorted({t for r in rows for t in [r["title"]] if RESP.search(t)})[:6]
    summary_rows.append({"accession": a, "n": len(rows), "all_keys": sorted(keys)[:14],
                         "resp_keys": resp_keys, "treat_keys": treat_keys, "titles_with_resp": title_resp})
for s in summary_rows:
    print("##", s["accession"], f"n={s['n']}")
    print("   keys:", s["all_keys"])
    if s["resp_keys"]: print("   RESPONSE:", {k: v for k, v in s["resp_keys"].items()})
    if s["treat_keys"]: print("   TREATMENT:", {k: v for k, v in s["treat_keys"].items()})
    if s["titles_with_resp"]: print("   titles:", s["titles_with_resp"])

KeyError: 'GSE269936'

#### Fetching remaining sample metadata and saning response fields

In [ ]:
rest = [a for a in c2.accession if a not in quick]
for a in rest:
    try:
        quick[a] = requests.get("https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi",
            params={"acc": a, "targ": "gsm", "form": "text", "view": "quick"}, timeout=240).text
    except Exception as e:
        quick[a] = "ERR " + repr(e)[:80]
    time.sleep(0.4)
json.dump(quick, open("handoff/gsm_quick.json","w"))

RESPKEY = re.compile(r"respon|recist|outcome|benefit|refractor|resist|progress|relaps|efficac|\bbor\b|survival", re.I)
scan = []
for a in c2.accession:
    if quick[a].startswith("ERR"): scan.append({"accession": a, "resp_fields": "FETCH_ERR", "values": ""}); continue
    rows = chars(a)
    keys = {}
    for r in rows:
        for cc in r["chars"]:
            k = cc.split(":")[0].strip().lower()
            v = cc.split(":",1)[1].strip() if ":" in cc else ""
            keys.setdefault(k, set()).add(v)
    hit = {k: sorted(v)[:10] for k, v in keys.items() if RESPKEY.search(k)}
    # also scan sample titles and descriptions
    tt = " ".join(r["title"] for r in rows)
    scan.append({"accession": a, "resp_fields": "; ".join(f"{k}={v}" for k, v in hit.items())[:220],
                 "title_hits": ";".join(sorted({w for w in re.findall(r"[A-Za-z_\-]*(?:respon|NR\b|_R_|resist)[A-Za-z_\-]*", tt, re.I)}))[:90]})
sc = pd.DataFrame(scan)
print(sc[(sc.resp_fields != "") | (sc.title_hits != "")].to_string(index=False))

accession          resp_fields title_hits  
GSE335480 clinical response=['NA', 'PD', 'PR', 'SD']  

### Fetching linked publications for shortlisted studies

In [15]:
pm = {a: [x for x in str(dfstu.set_index("accession").loc[a,"pubmed"]).split(";") if x.isdigit()] for a in keep16}
print({a: v for a, v in pm.items()})
allp = sorted({p for v in pm.values() for p in v})
ab = {}
if allp:
    r = eutil("efetch.fcgi", db="pubmed", id=",".join(allp), rettype="abstract", retmode="xml")
    root = ET.fromstring(r.text)
    for art in root.iter("PubmedArticle"):
        pid = art.findtext(".//PMID")
        ti = " ".join((art.findtext(".//ArticleTitle") or "").split())
        abst = " ".join(" ".join(x.itertext()) for x in art.iter("AbstractText"))
        jr = art.findtext(".//Journal/Title") or ""
        yr = art.findtext(".//PubDate/Year") or ""
        ab[pid] = {"title": ti, "journal": jr, "year": yr, "abstract": " ".join(abst.split())}
print(len(ab), "abstracts")



{'GSE269936': ['40890106'], 'GSE222446': ['38306416', '38714897'], 'GSE174401': ['34035069', '38030596'], 'GSE189889': ['35247927'], 'GSE222750': ['38306416', '38714897'], 'GSE111360': ['30550791'], 'GSE158803': ['34548323'], 'GSE186344': ['35063085'], 'GSE159251': ['33651880'], 'GSE242477': ['39630887', '40631515'], 'GSE221553': ['38306416', '38714897'], 'GSE139249': ['31801909'], 'GSE229858': ['39276771'], 'GSE261023': ['39276771'], 'GSE275929': ['39402027'], 'GSE179373': ['35584630', '35707680']}


NameError: name 'ET' is not defined

### Concepts

#### TIL - Tumor-Infiltrating Lymphocyte
#### ACT - Adoptive Cell Therapy

TIL-ACT stands for Tumor-Infiltrating Lymphocyte Adoptive Cell Therapy. It is a personalized cancer immunotherapy that extracts a patient's own immune cells from a tumor, multiplies them in a lab to number in the billions, and infuses them back into the body to attack the cancer. 

- **Tumor collection**: A surgeon removes a piece of the patient's tumor tissue. [
What is TIL therapy for cancer and how does it work? | Ohio State Medical Center]((https://www.youtube.com/watch?v=JdivYx9IR8E&t=1s))
- **Cell extraction and growth**: Technicians isolate the lymphocytes living inside the tumor and grow them in a lab using growth factors like IL-2. [Adoptive cell therapy with tumor-infiltrating lymphocytes in combination with nivolumab in patients with advanced melanoma](https://pmc.ncbi.nlm.nih.gov/articles/PMC11725098/), [Tumor-infiltrating lymphocyte (TIL) therapy](https://www.sciencedirect.com/science/chapter/edited-volume/abs/pii/B9780128226209000264)
- **Lymphodepletion**: The patient receives a short round of chemotherapy to clear space in the immune system for the new cells. [Tumor Infiltrating Lymphocyte (TIL) Therapy for Solid Tumor Treatment: Progressions and Challenges](https://pmc.ncbi.nlm.nih.gov/articles/PMC9455018/), [Tumor-Infiltrating Lymphocyte Therapy (TIL Therapy): Amplifying Your Body's Cancer-Fighting Cells - youtube](https://www.youtube.com/watch?v=sDB4GLBgurA)
- **Cell infusion**: Billions of the supercharged T-cells are returned to the patient's bloodstream to seek out and destroy tumors. [
What is TIL Therapy? - youtube - Dana_Farber](https://www.youtube.com/watch?v=QDe2rhtyLHM) and [
Tumor-Infiltrating Lymphocyte Therapy (TIL Therapy): Amplifying Your Body's Cancer-Fighting Cells - youtube - UChicago](https://www.youtube.com/watch?v=sDB4GLBgurA)
- **Immune stimulation**: Additional doses of IL-2 are given after the infusion to help the cells multiply and persist inside the body. [Optimizing TIL therapy for uveal melanoma: lessons learned and unlearned from cutaneous melanoma](https://pubmed.ncbi.nlm.nih.gov/40098478/) and [Adoptive cell therapy with tumor-infiltrating lymphocytes in combination with nivolumab in patients with advanced melanoma](https://pmc.ncbi.nlm.nih.gov/articles/PMC11725098/)

#### Key Uses

- Primarily studied and applied in solid tumors like metastatic melanoma.
- Expanding into clinical trials for other solid tumors like cervical cancer and lung cancer. [PragmaTIL Receives Approval to Advance Safer Cell Therapy for Cancer Patients](https://pragmatil.eu/2025/04/02/pragmatil-receives-approval-advance-safer-cell-therapy-cancer-patients/) and [Achievements and challenges of adoptive T cell therapy with tumor-infiltrating or blood-derived lymphocytes for metastatic melanoma: what is needed to achieve standard of care?](https://pmc.ncbi.nlm.nih.gov/articles/PMC11028895/)



### what are the differences between CART-Cell treatment and TIL-ACT?

Both are adoptive cell therapies — lymphodepletion followed by infusion of a large ex vivo–expanded autologous T-cell product — but they differ in where the cells come from, how they recognise the tumour, and consequently what they fail against.

| | TIL-ACT | CAR-T |
|---|---|---|
| Cell source | Lymphocytes resident in a resected tumour lesion | Peripheral blood T cells (leukapheresis) |
| Recognition | Native, unmodified TCRs → peptide–HLA complexes | Synthetic receptor: scFv ectodomain + hinge/TM + costim (CD28 or 4‑1BB) + CD3ζ → surface antigen, HLA-independent |
| Specificity breadth | Polyclonal, many antigens (neoantigens, cancer-germline, differentiation antigens); specificity is unknown at infusion | Monospecific, one defined antigen chosen in advance |
| Genetic modification | None (product is selected/expanded, not engineered) | Viral transduction or non-viral integration; requires vector manufacture, VCN and replication-competent-vector release testing |
| Manufacturing | Tumour fragments → pre-REP outgrowth in high-dose IL‑2 → rapid expansion with anti-CD3 and feeders; ~4–6 weeks, 10⁹–10¹¹ cells | Selection, activation, transduction, expansion; ~1–3 weeks to product release |
| Prerequisite | A resectable lesion that yields viable, expandable TIL | Adequate circulating T cells; antigen expressed on the tumour |
| Post-infusion support | Classically high-dose bolus IL‑2 | None |
| Dominant toxicities | IL‑2 toxicity (capillary leak, hypotension), lymphodepletion cytopenias; CRS uncommon and usually mild | CRS, ICANS, prolonged cytopenias, on-target/off-tumour injury, lineage aplasia (e.g. B-cell with CD19) |
| Where it works | Solid tumours — lifileucel received FDA accelerated approval in 2024 for unresectable/metastatic melanoma after anti-PD-1, the first TIL product licensed | B-cell malignancies and myeloma (CD19, BCMA); solid-tumour CAR-T remains investigational |

**The failure modes are mirror images.** CAR-T is blind to HLA, so β2M or HLA class I loss — a common melanoma immune-escape route — does not disable it; but antigen loss or downregulation does, and a single surface target must be near-uniformly expressed and absent from vital normal tissue, which is why solid tumours with heterogeneous, mostly intracellular antigens have resisted the approach. TIL recognises intracellular and neoantigen-derived peptides and hits many targets at once, so heterogeneity is less fatal, but every clone depends on intact antigen presentation, and the product's potency is not defined in advance — it is whatever tumour-reactive clonotypes happened to grow out.

**TCR-T sits between them:** blood-derived T cells engineered with a native-format TCR against a defined peptide–HLA pair (e.g. MAGE-A4 in synovial sarcoma). HLA-restricted like TIL, defined and manufacturable like CAR-T, and restricted to patients carrying the right HLA allele.

**Relevant to the datasets from earlier in this session:** the TIL-ACT series you now have (GSE222446, GSE221553, GSE229858, GSE261023) sample exactly the axis that makes TIL biology tractable — baseline tumour → infusion product → post-ACT tumour/blood — and because the receptor is native, scTCR-seq clonotypes serve as lineage barcodes linking pre-infusion tumour-resident clones to what persists and re-infiltrates. That is what makes them attractive for velocity work: you can ask which pre-infusion states give rise to the clones that expand in responders. A CAR-T equivalent does not exist in the melanoma GEO space, and would behave differently even if it did: the transgene is absent from the reference genome, so CAR⁺ cells need a custom reference to be detected at all, the native TCR is no longer a marker of tumour reactivity, and the construct itself carries no intron and therefore no spliced/unspliced signal for scVelo.

### Response-arm filter

I checked two sources per study: 
1. the per-sample GEO characteristics (scanned all 52 tumour-tissue series for response/RECIST/outcome fields) and 
2. the abstracts of the linked publications.

No study in the shortlist carries response labels in its GEO metadata. 

Across all 52 tumour-tissue series, exactly one does — GSE335480 (TIL-ACT, clinical response: PR / SD / PD per sample) — and its raw reads are not public, so it fails the scVelo criterion. For everything else, responder status has to be lifted from the paper and joined on the patient ID in the sample titles.

5 of 15 have both responders and non-responders profiled, across three cohorts:


| GEO	| Therapy	| Timepoints	| Velocity on	| Raw reads	| PMID | 
|-------|-----------|---------------|---------------|-----------|------|
| GSE269936	| anti-PD-1 ± anti-CTLA-4	| single, mostly pre-ICI	| malignant + immune	| 179 runs, 1.68 Tb	| 40890106 |
| GSE222446	| TIL-ACT phase 1 (prior ICI)	| pre-ACT (T0), 10 patients	| malignant + immune	| 13 runs, 0.22 Tb	| 38306416 |
| GSE221553	| TIL-ACT phase 1 (same cohort)	| pre- and post-ACT, | 13 patients	| CD45+ immune only	22 runs, 0.29 Tb	| 38306416 |
| GSE229858	| TIL-ACT	| ACT product, 13 patients	| T cells only	| 13 runs, 0.17 Tb	| 39276771 |
| GSE261023	| TIL-ACT	| baseline tumour + product, 2 patients	| T cells (multiome)	| 3 runs, 0.06 Tb	| 39276771 |


The **two TIL-ACT papers state the responder/non-responder contrast directly** — the Science Immunology study reports 
1. higher basal tumour-intrinsic immunogenicity and 
2. mutational burden in ACT responders with distinct CD8 TIL and myeloid states, and 
3. the Immunity study tracks clonotypes from baseline tumour through product to post-ACT blood and tumour, 
4. contrasting patients with and without clinical response. 
GSE269936 stratifies on PFS under ICI rather than RECIST categories, so its "+/−" axis is time-to-progression, not a binary label.

Two partial matches: GSE174401 (mixed systemic therapy, stratified by overall survival with one long-term survivor against poor survivors) and GSE139249 (NK cells including lesions from patients who had progressed on checkpoint blockade — progressors only, no responder arm). The remaining 8 are treatment-naive cohorts, atlases, a radiosurgery/dexamethasone trial (GSE275929 — its two arms are steroid dose, not immunotherapy response), or ex vivo ICB modelling on organoids (GSE111360).

If the goal is velocity in malignant cells with a response contrast, the field narrows to GSE269936 and GSE222446. For the Lausanne cohort, GSE222446 (unsorted 10x 3′ GEX, pre-ACT) and GSE221553 (CD45+ 5′ GEX, pre + post-ACT) are the same 13 patients, with patient1…patient13 IDs in both the titles and the paper's supplementary tables, so responder labels transfer cleanly and the two can be analysed as one cohort.

One correction to the previous shortlist: GSE222750 is scTCR-seq only (cellranger vdj, contig annotation files, no gene-expression libraries) — its GEX counterpart is GSE222446. It is now moved to the exclusions, so the scVelo-ready set is 15 studies, not 16.

1. melanoma_scvelo_immunotherapy_response.csv — all 15 studies with therapy, response-arm verdict, the evidence sentence behind each call, where the labels live, timepoints and run counts
2. melanoma_scvelo_ready_studies.csv — corrected 15-study shortlist (v2)
3. melanoma_scvelo_excluded_with_reasons.csv — now 37 rows including GSE222750 (v2)


In [16]:
fname = 'melanoma_scvelo_ready_studies.csv'
dfstu = pd.read_csv(os.path.join(root_result, fname))
print(dfstu.shape)
print(dfstu.columns)
dfstu.head(3)

(16, 20)
Index(['accession', 'tier', 'cutaneous', 'year', 'context', 'specimen',
       'sorting', 'scvelo_target', 'melanoma_fraction', 'platform', 'n_gsm',
       'rna_runs_public', 'raw_Tbases', 'read_structure', 'fastq_route',
       'note', 'bioproject', 'pubmed', 'geo_url', 'title'],
      dtype='object')


,accession,tier,cutaneous,year,context,specimen,sorting,scvelo_target,melanoma_fraction,platform,n_gsm,rna_runs_public,raw_Tbases,read_structure,fastq_route,note,bioproject,pubmed,geo_url,title
0,GSE269936,B_site_not_stated,implied,2025,"Metastatic melanoma, scRNA-seq, heterogeneity ...",metastasis_lymph_node;metastasis_other,whole_tumor_unsorted,malignant + immune,all,unclear,38,179,1.677,"R0:8,R1:26,R2:57",SRA only: fasterq-dump --include-technical --s...,Whole dissociated metastatic melanoma tumor ce...,PRJNA1124138,40890106,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Gene expression profile at single cell level f...
1,GSE222446,B_site_not_stated,implied,2024,"Metastatic melanoma, pre/post TIL adoptive cel...",metastasis_other,whole_tumor_unsorted,malignant + immune,all,10x_3prime,12,13,0.224,"R0:28,R1:91,R2:10,R3:10",SRA only: fasterq-dump --include-technical --s...,Dissociated whole-tumor scRNA-seq from metasta...,PRJNA922121,38306416;38714897,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Response to tumor- infiltrating lymphocyte ado...
2,GSE174401,A_cutaneous_confirmed,implied,2021,"Melanoma brain/leptomeningeal metastases, CSF ...",metastasis_brain;metastasis_other,whole_tumor_unsorted,malignant + immune,all,unclear,43,280,0.198,"R0:8,R1:26,R2:98",SRA only: fasterq-dump --include-technical --s...,CSF and brain/skin metastasis cells; immune mi...,PRJNA729750,34035069;38030596,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Single cell characterization of the immune mic...


In [18]:
fname = 'melanoma_scvelo_ready_studies.csv'
dfstu = pd.read_csv(os.path.join(root_result, fname))
print(dfstu.shape)
print(dfstu.columns)
dfstu.head(3)

(16, 20)
Index(['accession', 'tier', 'cutaneous', 'year', 'context', 'specimen',
       'sorting', 'scvelo_target', 'melanoma_fraction', 'platform', 'n_gsm',
       'rna_runs_public', 'raw_Tbases', 'read_structure', 'fastq_route',
       'note', 'bioproject', 'pubmed', 'geo_url', 'title'],
      dtype='object')


,accession,tier,cutaneous,year,context,specimen,sorting,scvelo_target,melanoma_fraction,platform,n_gsm,rna_runs_public,raw_Tbases,read_structure,fastq_route,note,bioproject,pubmed,geo_url,title
0,GSE269936,B_site_not_stated,implied,2025,"Metastatic melanoma, scRNA-seq, heterogeneity ...",metastasis_lymph_node;metastasis_other,whole_tumor_unsorted,malignant + immune,all,unclear,38,179,1.677,"R0:8,R1:26,R2:57",SRA only: fasterq-dump --include-technical --s...,Whole dissociated metastatic melanoma tumor ce...,PRJNA1124138,40890106,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Gene expression profile at single cell level f...
1,GSE222446,B_site_not_stated,implied,2024,"Metastatic melanoma, pre/post TIL adoptive cel...",metastasis_other,whole_tumor_unsorted,malignant + immune,all,10x_3prime,12,13,0.224,"R0:28,R1:91,R2:10,R3:10",SRA only: fasterq-dump --include-technical --s...,Dissociated whole-tumor scRNA-seq from metasta...,PRJNA922121,38306416;38714897,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Response to tumor- infiltrating lymphocyte ado...
2,GSE174401,A_cutaneous_confirmed,implied,2021,"Melanoma brain/leptomeningeal metastases, CSF ...",metastasis_brain;metastasis_other,whole_tumor_unsorted,malignant + immune,all,unclear,43,280,0.198,"R0:8,R1:26,R2:98",SRA only: fasterq-dump --include-technical --s...,CSF and brain/skin metastasis cells; immune mi...,PRJNA729750,34035069;38030596,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Single cell characterization of the immune mic...


In [19]:
fname = 'melanoma_scvelo_immunotherapy_response.csv'
dfstu = pd.read_csv(os.path.join(root_result, fname))
print(dfstu.shape)
print(dfstu.columns)
dfstu.head(3)

(15, 20)
Index(['accession', 'response_arms', 'therapy', 'treatment_timepoints',
       'response_labels', 'arms_evidence', 'scvelo_target', 'sorting', 'tier',
       'cutaneous', 'melanoma_fraction', 'context', 'specimen',
       'rna_runs_public', 'raw_Tbases', 'read_structure', 'fastq_route',
       'cohort_pmid', 'bioproject', 'geo_url'],
      dtype='object')


,accession,response_arms,therapy,treatment_timepoints,response_labels,arms_evidence,scvelo_target,sorting,tier,cutaneous,melanoma_fraction,context,specimen,rna_runs_public,raw_Tbases,read_structure,fastq_route,cohort_pmid,bioproject,geo_url
0,GSE269936,yes,anti-PD-1 ± anti-CTLA-4 (ICI),single timepoint (mostly pre-ICI),paper_supplementary,36 metastatic melanoma samples; mregDC abundan...,malignant + immune,whole_tumor_unsorted,B_site_not_stated,implied,all,"Metastatic melanoma, scRNA-seq, heterogeneity ...",metastasis_lymph_node;metastasis_other,179,1.677,"R0:8,R1:26,R2:57",SRA only: fasterq-dump --include-technical --s...,40890106,PRJNA1124138,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
1,GSE222446,yes,TIL adoptive cell therapy (phase 1) + prior ICI,"pre-ACT (T0) only, 10 patients",paper_supplementary,paper contrasts ACT responders vs nonresponder...,malignant + immune,whole_tumor_unsorted,B_site_not_stated,implied,all,"Metastatic melanoma, pre/post TIL adoptive cel...",metastasis_other,13,0.224,"R0:28,R1:91,R2:10,R3:10",SRA only: fasterq-dump --include-technical --s...,38306416,PRJNA922121,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...
2,GSE221553,yes,TIL adoptive cell therapy (phase 1),"pre- and post-ACT, 13 patients",paper_supplementary,"same 13-patient TIL-ACT cohort, responder vs n...",immune cells only,cd45_immune_sorted,B_site_not_stated,implied,all,"Metastatic melanoma, TIL-ACT therapy, pre/post...",metastasis_other,22,0.291,"R0:8,R1:26,R2:91",SRA only: fasterq-dump --include-technical --s...,38306416,PRJNA914772,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...


In [20]:
fname = 'melanoma_scvelo_excluded_with_reasons.csv'
dfstu = pd.read_csv(os.path.join(root_result, fname))
print(dfstu.shape)
print(dfstu.columns)
dfstu.head(3)

(37, 8)
Index(['accession', 'tier', 'year', 'context', 'specimen', 'platform',
       'runs_with_fastq', 'excluded_because'],
      dtype='object')


,accession,tier,year,context,specimen,platform,runs_with_fastq,excluded_because
0,GSE242941,A_cutaneous_confirmed,2026,"Acral melanoma and other skin cancer NK cells,...",primary_skin_tumor,10x_3prime,0,no public raw reads (processed matrices only /...
1,GSE294274,A_cutaneous_confirmed,2026,Metastatic melanoma lymph nodes; ICI-naive/res...,metastasis_lymph_node,10x_5prime,0,no public raw reads (processed matrices only /...
2,GSE335480,A_cutaneous_confirmed,2026,"TIL-ACT for metastatic melanoma, longitudinal ...",primary_skin_tumor;metastasis_lymph_node;metas...,10x_5prime,0,no public raw reads (processed matrices only /...
